In [ ]:
SCAN_SET_ID = "ksr216_2026apr_repeats_01"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")

In [ ]:
from pprint import pprint

from rsoxs_unified_workup.config import load_scan_set
from rsoxs_unified_workup.loading import (
    build_base_description,
    scan_ids_by_polarization,
)

configuration = load_scan_set(SCAN_SET_ID)

print(build_base_description(configuration["sample"]))
pprint(scan_ids_by_polarization(configuration))

In [ ]:
import httpx
import PyHyperScattering as phs
from tiled.client import from_uri

catalog = from_uri(
    "https://tiled.nsls2.bnl.gov",
    timeout=httpx.Timeout(None),
)

loader = phs.load.SST1RSoXSDB(
    corr_mode="none",
    catalog=catalog["rsoxs"]["raw"],
)

In [ ]:
from rsoxs_unified_workup.loading import load_scans

scan = load_scans(loader, configuration)

for polarization, scan_data in scan.items():
    print(
        f"Scan {scan_data['scan_id']}: "
        f"configured={polarization}°, "
        f"measured={scan_data['data'].attrs['polarization']}°"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_raw_frames

plot_raw_frames(scan)

In [ ]:
from rsoxs_unified_workup.darks import load_dark_frames

scan = load_dark_frames(loader, scan)

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_dark_histograms,
)

dark_upper_percentile = configuration[
    "processing"
]["darks"]["upper_percentile"]

dark_cutoffs = plot_dark_histograms(
    scan,
    upper_percentile=dark_upper_percentile,
)

{
    polarization: round(float(cutoff), 2)
    for polarization, cutoff in dark_cutoffs.items()
}

In [ ]:
from rsoxs_unified_workup.darks import build_dark_masks

dark_masks, dark_mask_summaries = build_dark_masks(
    scan,
    dark_cutoffs,
)

for polarization, summary in dark_mask_summaries.items():
    print(
        f"Scan {summary['scan_id']}, "
        f"polarization={polarization}°: "
        f"cutoff={summary['cutoff']:.2f}, "
        f"masked pixels={summary['masked_pixels']:,} "
        f"({summary['masked_percent']:.4f}%)"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_dark_masks

plot_dark_masks(scan, dark_masks)

In [ ]:
from rsoxs_unified_workup.masks import (
    combine_static_masks,
    create_beamstop_mask,
    create_border_mask,
)

processing = configuration["processing"]
detector_shape = next(
    iter(scan.values())
)["data"].shape[-2:]

beamstop_mask, beamstop_vertices = create_beamstop_mask(
    detector_shape,
    processing["beamstop"],
)

border_mask = create_border_mask(
    detector_shape,
    processing["detector_borders"],
)

static_masks = combine_static_masks(
    dark_masks,
    beamstop_mask,
    border_mask,
    union_across_polarizations=processing[
        "darks"
    ]["union_across_polarizations"],
)

print("Beamstop vertices:")
for x, y in beamstop_vertices:
    print(f"  ({x:.2f}, {y:.2f})")

print("\nTotal masked pixels:")
for polarization, mask in static_masks.items():
    print(
        f"  polarization={polarization}°: "
        f"{int(mask.sum().item()):,}"
    )

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_static_mask_overlays,
)

plot_static_mask_overlays(
    scan,
    static_masks,
    beamstop_vertices,
)


In [ ]:
from rsoxs_unified_workup.masks import (
    apply_static_masks,
    build_saturation_masks,
)

scan = apply_static_masks(scan, static_masks)
saturation_masks = build_saturation_masks(scan)

for index, energy in enumerate(
    saturation_masks.energy.values
):
    mask = saturation_masks.isel(system=index)
    print(
        f"Energy={energy:.1f} eV: "
        f"{int(mask.sum().item()):,} "
        f"saturated pixels across all polarizations"
    )

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_saturation_overlays,
)

plot_saturation_overlays(
    scan,
    saturation_masks,
)

In [ ]:
from rsoxs_unified_workup.masks import (
    build_center_masks,
)

center_masks, center_mask_summaries = build_center_masks(
    saturation_masks,
    beamstop_vertices,
    configuration["processing"]["center_mask"],
)

for summary in center_mask_summaries:
    print(
        f"Energy={summary['energy']:.1f} eV: "
        f"radius={summary['radius']:.2f} pixels, "
        f"source={summary['radius_source']}, "
        f"farthest point="
        f"({summary['farthest_x']:.2f}, "
        f"{summary['farthest_y']:.2f})"
    )


In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_center_mask_review,
)

plot_center_mask_review(
    scan,
    saturation_masks,
    center_masks,
    center_mask_summaries,
    configuration["processing"]["center_mask"],
)

In [ ]:
from rsoxs_unified_workup.masks import (
    apply_final_masks,
    build_final_masks,
)

final_masks = build_final_masks(
    static_masks,
    saturation_masks,
    center_masks,
)

scan = apply_final_masks(scan, final_masks)

for polarization, final_mask in final_masks.items():
    mask_2d = final_mask.isel(system=0).transpose(
        "pix_y",
        "pix_x",
    )

    print(
        f"Polarization={polarization}°: "
        f"full={final_mask.dims} {final_mask.shape}; "
        f"energy 0={mask_2d.dims} {mask_2d.shape}"
    )

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_final_masked_frames,
)

plot_final_masked_frames(
    scan,
    configuration["processing"]["center_mask"],
)

In [ ]:
import rsoxs_unified_workup.beam_center as beam_center

fit_results, median_center_suggestion = (
    beam_center.fit_scan_set_beam_centers(
        scans=scan,
        final_masks=final_masks,
        processing=configuration["processing"],
    )
)

for polarization, result in fit_results.items():
    center = result["fine_best"]
    print(
        f"{polarization}°: "
        f"({center['x']}, {center['y']}), "
        f"cost={center['cost']:.6e}"
    )

print(
    "Median suggestion:",
    median_center_suggestion,
)

In [ ]:
import rsoxs_unified_workup.plotting as workup_plotting

for polarization, result in fit_results.items():
    workup_plotting.plot_beam_center_fit(
        result,
        scan[polarization],
    )

After reviewing all four fits, manually add the accepted center under
[shared_beam_center] in the processing TOML. The median is a suggestion only.

In [ ]:
shared_beam_center = beam_center.get_shared_beam_center(
    configuration["processing"]
)

print("Median suggestion:", median_center_suggestion)
print("Reviewed shared center:", shared_beam_center)
print(
    "Difference from median:",
    {
        "x": (
            shared_beam_center["x"]
            - median_center_suggestion["x"]
        ),
        "y": (
            shared_beam_center["y"]
            - median_center_suggestion["y"]
        ),
    },
)

In [ ]:
import rsoxs_unified_workup.integration as integration

integration_results = integration.integrate_scan_set(
    scans=scan,
    final_masks=final_masks,
    processing=configuration["processing"],
)

for polarization, result in integration_results.items():
    print(
        f"{polarization}°: "
        f"clean shape={result['clean'].shape}; "
        f"q cap={result['q_cap'].values}"
    )


In [ ]:
for polarization, result in integration_results.items():
    workup_plotting.plot_integration_review(
        scan_record=scan[polarization],
        result=result,
    )

In [ ]:
dark_integration_results = (
    integration.integrate_dark_scan_set(
        scans=scan,
        final_masks=final_masks,
        integration_results=integration_results,
        processing=configuration["processing"],
    )
)

for polarization, result in (
    dark_integration_results.items()
):
    print(
        f"{polarization}°: "
        f"dark frame={result['dark_frame_index']}; "
        f"shape={result['darks_clean'].shape}"
    )

In [ ]:
for polarization, result in (
    dark_integration_results.items()
):
    workup_plotting.plot_dark_integration_review(
        scan_record=scan[polarization],
        result=result,
        upper_percentile=configuration[
            "processing"
        ]["darks"]["upper_percentile"],
    )

In [ ]:
import rsoxs_unified_workup.normalization as normalization

normalization_results = normalization.normalize_scan_set(
    integration_results=integration_results,
    processing=configuration["processing"],
)

for polarization, result in normalization_results.items():
    full_check = result[
        "full_ring_normalized"
    ].mean("chi", skipna=True)

    half_region = normalization.select_chi_region(
        result["half_ring_normalized"],
        center_degrees=result[
            "half_ring_center_degrees"
        ],
        halfwidth_degrees=result[
            "half_ring_halfwidth_degrees"
        ],
    )
    half_check = half_region.mean(
        "chi",
        skipna=True,
    )

    full_deviation = np.abs(full_check - 1).max(
        skipna=True
    ).item()
    half_deviation = np.abs(half_check - 1).max(
        skipna=True
    ).item()

    print(
        f"{polarization}°: "
        f"full-ring maximum deviation={full_deviation:.3e}; "
        f"half-ring maximum deviation={half_deviation:.3e}"
    )


In [ ]:
workup_plotting.plot_normalization_review(
    scans=scan,
    normalization_results=normalization_results,
)

In [ ]:
flux_correction_results = (
    normalization.flux_correct_scan_set(
        scans=scan,
        integration_results=integration_results,
        processing=configuration["processing"],
    )
)

for polarization, result in (
    flux_correction_results.items()
):
    flux = result["flux"]
    print(
        f"{polarization}°: "
        f"energies={flux.energy.values}; "
        f"Au mesh current={flux.values}; "
        f"284.8/286 ratio="
        f"{flux.values[0] / flux.values[1]:.6f}"
    )


In [ ]:
workup_plotting.plot_flux_correction_review(
    scans=scan,
    integration_results=integration_results,
    flux_results=flux_correction_results,
)

In [ ]:
import rsoxs_unified_workup.anisotropy as anisotropy

anisotropy_results = anisotropy.calculate_anisotropy(
    configuration=configuration,
    normalization_results=normalization_results,
)

for sector in anisotropy_results["full_ring"].values():
    values = sector["anisotropy"]
    print(
        f"{sector['label']}: "
        f"parallel={sector['parallel_polarization']}°, "
        f"perpendicular="
        f"{sector['perpendicular_polarization']}°; "
        f"shape={values.shape}; "
        f"finite={np.isfinite(values).sum().item()}"
    )

In [ ]:
workup_plotting.plot_anisotropy_review(
    scans=scan,
    anisotropy_results=anisotropy_results,
)

In [ ]:
workup_plotting.plot_anisotropy_sector_overlays(
    scans=scan,
    configuration=configuration,
    anisotropy_results=anisotropy_results,
    energy_value=configuration["processing"][
        "beam_center_fit"
    ]["fit_energy"],
)

In [ ]:
aggregate_anisotropy_results = (
    anisotropy.aggregate_anisotropy(
        anisotropy_results
    )
)

for mode, results in aggregate_anisotropy_results.items():
    print(f"\n{mode}")

    for method_name in [
        "method_1_global_intensity_ratio",
        "method_2_pointwise_anisotropy",
        "method_3_sector_anisotropy_mean",
    ]:
        values = results[method_name]

        print(
            f"  {method_name}: "
            f"dims={values.dims}, "
            f"shape={values.shape}, "
            f"finite={np.isfinite(values.values).sum()}"
        )

In [ ]:
workup_plotting.plot_aggregate_anisotropy_comparison(
    scans=scan,
    aggregate_results=aggregate_anisotropy_results,
)

In [ ]:
pointwise_anisotropy_results = (
    anisotropy.build_pointwise_anisotropy_maps(
        anisotropy_results
    )
)

for mode, pair_results in (
    pointwise_anisotropy_results.items()
):
    print(f"\n{mode}")

    for pair_id, result in pair_results.items():
        values = result["anisotropy"]

        print(
            f"  {pair_id}: "
            f"dims={values.dims}, "
            f"shape={values.shape}, "
            f"chi range="
            f"{float(values.chi.min()):g} to "
            f"{float(values.chi.max()):g}; "
            f"finite="
            f"{np.isfinite(values.values).sum()}"
        )


In [ ]:
workup_plotting.plot_pointwise_anisotropy_review(
    scans=scan,
    pointwise_results=pointwise_anisotropy_results,
)